The action grid is

0 1 2

3 4 5

6 7 8

entering 2 terminates and gives +1, entering 8 terminates and gives +10

The policy can move down left up or right and so when we run the experiment with a gamma of .2 it prioritizes moving towards the nearby terminal state at 2 but when we use a larger gamma of .9 it recognizes that moving to 8 yields a higher reward. This shows how the gamma value controls how it changes the policy in a way that prioritizes short term greedy gains or long term higher value payoffs.

--- Experiment: gamma = 0.9 ---

Value Function (gamma = 0.9):

7.29   8.10   0.00

8.10   9.00   10.00

9.00   10.00   0.00


Policy (gamma = 0.9):

↓   ↓   T

↓   ↓   ↓

→   →   T

--- Experiment: gamma = 0.2 ---

Value Function (gamma = 0.2):

0.20   1.00   0.00

0.40   2.00   10.00

2.00   10.00   0.00

Policy (gamma = 0.2):

→   →   T

↓   ↓   ↓

→   →   T

In [2]:
import numpy as np

# 3x3 grid world

n_states = 9
n_actions = 4

positions = [(i // 3, i % 3) for i in range(n_states)]

#compute movement to next state
next_state = np.zeros((n_states, n_actions), dtype=int)
def get_next_state(s, a):
    r, c = positions[s]
    dr, dc = [(-1, 0), (1, 0), (0, -1), (0, 1)][a]
    nr, nc = r + dr, c + dc
    if 0 <= nr < 3 and 0 <= nc < 3:
        return nr * 3 + nc
    return s
for s in range(n_states):
    for a in range(n_actions):
        next_state[s, a] = get_next_state(s, a)

#reward movement into the goal states
def get_reward(s, a):
    s_prime = next_state[s, a]
    if s_prime == 2 and s != 2:
        return 1.0
    elif s_prime == 8 and s != 8:
        return 10.0
    return 0.0

def value_iteration(gamma=0.9, theta=1e-6):
    V = np.zeros(n_states)
    iteration = 0
    while True:
        delta = 0
        #Iterate over states
        for s in range(n_states):
            v = V[s]
            #terminal states
            if s in [2, 8]:
                V[s] = 0.0
                delta = max(delta, abs(v - V[s]))
                continue

            max_val = float('-inf')
            #iterate over actions and find the state value
            for a in range(n_actions):
                s_prime = next_state[s, a]
                r = get_reward(s, a)
                val = r + gamma * V[s_prime]
                if val > max_val:
                    max_val = val
            V[s] = max_val
            delta = max(delta, abs(v - V[s]))
        iteration += 1
        if delta < theta:
            break
    return V

#get the optimal greedy policy
def extract_policy(V, gamma):
    policy = np.full(n_states, -1, dtype=int)
    for s in range(n_states):
        if s in [2, 8]:
            continue
        max_val = float('-inf')
        best_a = 0
        for a in range(n_actions):
            s_prime = next_state[s, a]
            r = get_reward(s, a)
            val = r + gamma * V[s_prime]
            if val > max_val:
                max_val = val
                best_a = a
        policy[s] = best_a
    return policy

def print_value_function(V, gamma):
    for i in range(3):
        row = [f'{V[i*3 + j]:.2f}' for j in range(3)]
        print('   '.join(row))

def print_policy(policy, gamma):
    action_names = ['↑', '↓', '←', '→']
    for i in range(3):
        row = []
        for j in range(3):
            s = i * 3 + j
            if s in [2, 8]:
                row.append('T')
            else:
                row.append(action_names[policy[s]])
        print('   '.join(row))

for gamma in [0.9, 0.2]:
    print("Gamma:",gamma)
    V = value_iteration(gamma)
    policy = extract_policy(V, gamma)
    print_value_function(V, gamma)
    print_policy(policy, gamma)

Gamma: 0.9
7.29   8.10   0.00
8.10   9.00   10.00
9.00   10.00   0.00
↓   ↓   T
↓   ↓   ↓
→   →   T
Gamma: 0.2
0.20   1.00   0.00
0.40   2.00   10.00
2.00   10.00   0.00
→   →   T
↓   ↓   ↓
→   →   T


This is a Markov Decision Process because it has a set of states which is the current position in the grid. The actions the actor can take are up down left and right. The transition function is defined by the next_state values and the reward funciton gives a reward based on only the current state and action/resulting next state. Additionally the discount factor is 0.9 which is between 0 and 1.